In [11]:
import cv2
import torch
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.nn as nn

img_real = cv2.imread('cachorroreal.jpg')
img_fake = cv2.imread('cachorrofake.png')

img_real = cv2.cvtColor(img_real, cv2.COLOR_BGR2RGB)
img_fake = cv2.cvtColor(img_fake, cv2.COLOR_BGR2RGB)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((32, 32)),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

img_real_tensor = transform(img_real).unsqueeze(0).to('cuda') 
img_fake_tensor = transform(img_fake).unsqueeze(0).to('cuda')

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [14]:
model_cifake = SimpleCNN().to('cuda')
model_cifake.load_state_dict(torch.load('simple_cnn_cifake_model.pth', map_location='cuda'))
model_cifake.eval()

model_kaggle = SimpleCNN().to('cuda')
model_kaggle.load_state_dict(torch.load('simple_cnn_kaggle_model.pth', map_location='cuda'))
model_kaggle.eval()

with torch.no_grad():
    output_real_cifake = model_cifake(img_real_tensor)
    output_fake_cifake = model_cifake(img_fake_tensor)
    output_real_kaggle = model_kaggle(img_real_tensor)
    output_fake_kaggle = model_kaggle(img_fake_tensor)

    prob_real_cifake = F.softmax(output_real_cifake, dim=1)
    prob_fake_cifake = F.softmax(output_fake_cifake, dim=1)
    prob_real_kaggle = F.softmax(output_real_kaggle, dim=1)
    prob_fake_kaggle = F.softmax(output_fake_kaggle, dim=1)

    confianca_real_cifake, class_real_cifake = torch.max(prob_real_cifake, 1)
    confianca_fake_cifake, class_fake_cifake = torch.max(prob_fake_cifake, 1)
    confianca_real_kaggle, class_real_kaggle = torch.max(prob_real_kaggle, 1)
    confianca_fake_kaggle, class_fake_kaggle = torch.max(prob_fake_kaggle, 1)

    print(f"CIFake - Real: Classe {class_real_cifake.item()} com confiança {confianca_real_cifake.item():.4f}")
    print(f"CIFake - Fake: Classe {class_fake_cifake.item()} com confiança {confianca_fake_cifake.item():.4f}")
    print("-------------------------------------------------------------------------")
    print(f"Kaggle - Real: Classe {class_real_kaggle.item()} com confiança {confianca_real_kaggle.item():.4f}")
    print(f"Kaggle - Fake: Classe {class_fake_kaggle.item()} com confiança {confianca_fake_kaggle.item():.4f}")

CIFake - Real: Classe 1 com confiança 0.5609
CIFake - Fake: Classe 1 com confiança 0.9979
-------------------------------------------------------------------------
Kaggle - Real: Classe 1 com confiança 0.6321
Kaggle - Fake: Classe 1 com confiança 0.7358


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score

model_kaggle.eval()

cifake_path = './cifake/test'
cifake_dataset = ImageFolder(root=cifake_path, transform=transform)
cifake_loader = DataLoader(cifake_dataset, batch_size=64, shuffle=False)

all_preds = []
all_labels = []


with torch.no_grad():
    for inputs, labels in cifake_loader:
        inputs, labels = inputs.to('cuda'), labels.to('cuda')

        output = model_kaggle(inputs)
        _, predicted = torch.max(output, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Accurarcy: {acc * 100:.2f}%\n")

print("Report by Class:")
print(classification_report(all_labels, all_preds, target_names=cifake_dataset.classes))

Accurarcy: 45.04%

Report by Class:
              precision    recall  f1-score   support

        FAKE       0.37      0.14      0.20     10000
        REAL       0.47      0.76      0.58     10000

    accuracy                           0.45     20000
   macro avg       0.42      0.45      0.39     20000
weighted avg       0.42      0.45      0.39     20000



In [19]:
kaggle_path = './deepfake_kaggle/Test'
kaggle_dataset = ImageFolder(root=kaggle_path, transform=transform)
kaggle_loader = DataLoader(kaggle_dataset, batch_size=64, shuffle=False)

model_kaggle.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in kaggle_loader:
        inputs, labels = inputs.to('cuda'), labels.to('cuda')

        output = model_cifake(inputs)
        _, predicted = torch.max(output, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Test Accurarcy: {acc * 100:.2f}%\n")

print("Report by Class:")
print(classification_report(all_labels, all_preds, target_names=kaggle_dataset.classes))

Test Accurarcy: 49.28%

Report by Class:
              precision    recall  f1-score   support

        Fake       0.44      0.02      0.05      5492
        Real       0.49      0.97      0.65      5413

    accuracy                           0.49     10905
   macro avg       0.47      0.50      0.35     10905
weighted avg       0.47      0.49      0.35     10905

